スクレピング前に確認

In [1]:
import requests
from numpy.core.defchararray import title

url = "https://example.com/robots.txt"

response = requests.get(url)

print(response.text)

<!doctype html><html lang="en"><head><title>Example Domain</title><meta name="viewport" content="width=device-width, initial-scale=1"><style>body{background:#eee;width:60vw;margin:15vh auto;font-family:system-ui,sans-serif}h1{font-size:1.5em}div{opacity:0.8}a:link,a:visited{color:#348}</style></head><body><div><h1>Example Domain</h1><p>This domain is for use in documentation examples without needing permission. Avoid use in operations.</p><p><a href="https://iana.org/domains/example">Learn more</a></p></div></body></html>



課題
- 日経新聞のネットメディアのページから、記事のタイトルとURLを取得する

In [2]:
import requests
from bs4 import BeautifulSoup
page = 1
url = "https://www.nikkei.com/business/net-media/?page={}".format(page)
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")



In [3]:
links = soup.find_all("a", class_="fauxBlockLink_f6t5v6i")
link_page_one = soup.find_all("a", class_="fauxBlockLink_fzas0ps")
links.insert(0, link_page_one[0])


print(links)

[<a class="fauxBlockLink_fzas0ps" href="/article/DGXZQOUC05AJR0V00C26A3000000/">ウェザーニューズ、20万人の「天気愛」をお茶の間に</a>, <a class="fauxBlockLink_f6t5v6i" href="/article/DGXZQOUB092PE0Z00C26A5000000/">みずほが楽天銀行出資へ　5〜10%軸、楽天子会社再編で資本付け替え</a>, <a class="fauxBlockLink_f6t5v6i" href="/article/DGXZQOUC15AXI0V10C26A5000000/">「AI吹き替え」で感情や声質も再現　200言語に対応、新興がサービス</a>, <a class="fauxBlockLink_f6t5v6i" href="/article/DGXZQOUC167JL0W6A310C2000000/">宣材動画の制作10分で　ソフトバンク、生成AIで作成代行</a>, <a class="fauxBlockLink_f6t5v6i" href="/article/DGXZQOGM161730W6A510C2000000/">TSMC熊本工場、初の最終黒字　26年1〜3月期</a>, <a class="fauxBlockLink_f6t5v6i" href="/article/DGXZQOUC158P10V10C26A5000000/">Claude Mythosだけじゃない　AIサイバー攻撃が暴く未知のバグ</a>, <a class="fauxBlockLink_f6t5v6i" href="/article/DGXZQOCD2826H0Y6A420C2000000/">漫画家やイラストレーター「AI普及で収入減」2割　著作権保護求める声</a>, <a class="fauxBlockLink_f6t5v6i" href="/article/DGXZQOUC155LV0V10C26A4000000/">ソフトバンク系､炒め物特化の調理ロボ　冷蔵庫から自動で食材</a>, <a class="fauxBlockLink_f6t5v6i" href="/article/DGXZQOUC155830V10C26A5000000

In [4]:
titles = []

for link in links:
    #U3000除く
    titles.append(link.get_text().replace("\u3000", ""))

In [5]:
print(titles)

['ウェザーニューズ、20万人の「天気愛」をお茶の間に', 'みずほが楽天銀行出資へ5〜10%軸、楽天子会社再編で資本付け替え', '「AI吹き替え」で感情や声質も再現200言語に対応、新興がサービス', '宣材動画の制作10分でソフトバンク、生成AIで作成代行', 'TSMC熊本工場、初の最終黒字26年1〜3月期', 'Claude MythosだけじゃないAIサイバー攻撃が暴く未知のバグ', '漫画家やイラストレーター「AI普及で収入減」2割著作権保護求める声', 'ソフトバンク系､炒め物特化の調理ロボ冷蔵庫から自動で食材', 'GMO純利益14%増1〜3月、キャッシュレス決済やネット金融好調', 'エレコムの営業利益6%増27年3月期、純利益は反動減', '携帯契約時のポイント特典、｢最長1年で分割｣なら許容総務省方針', 'NEC、施設管理子会社の売却検討1000億円規模', 'Claude Mythosとはシステムの未知の脆弱性検知、悪用の恐れ', '関西スタートアップの資金調達を支援VCが新組織、地銀など参画', '富士通と日本IBM、医療分野でデータ連携AIで効率化', 'サイバーエージェント、京都にクリエーター育成拠点人材不足に対応', 'KDDIと楽天モバイル、通信設備の省電力技術を共同開発へ', 'NEC、太平洋島しょ国を結ぶ光海底ケーブルの建設完了', 'NEC、米通信向けソフト会社の買収完了', '川崎重工、船の係留ロープ管理システムを実証NSユナイテッドと']


In [13]:
#本文取得修正
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

DOMAIN = "https://www.nikkei.com/"
body_texts = []

headers = {
    "User-Agent": "Mozilla/5.0"
}

for link in links:
    href = link.get("href")

    if not href:
        body_texts.append("")
        continue

    body_link = urljoin(DOMAIN, href)

    body_response = requests.get(body_link, headers=headers)
    body_soup = BeautifulSoup(body_response.text, "html.parser")

    paragraphs = body_soup.find_all("p", class_="paragraph_p18mfke4")

    if not paragraphs:
        print("本文が見つかりません:", body_link)
        body_texts.append("")
        continue

    body_text = "\n".join(p.get_text(strip=True) for p in paragraphs)
    body_texts.append(body_text)

print(body_texts)

本文が見つかりません: https://www.nikkei.com/article/DGXZQOCD2826H0Y6A420C2000000/
['天候不順や激甚災害が増えるなか、天気情報への関心はこれまでになく高まっている。高まる関心を生かして正確な情報発信につなげるべく、ウェザーニューズはテレビ局向けに新サービスを始めた。1日約20万の投稿から真偽を見極めた情報を、お茶の間に届ける。\n企業向け「ウェザーニュース\u3000for business」のオプションとして、テレビ局向けの放送業務支援サービスを始めた。雨雲レーダーや台風進路図など20種類...', 'みずほフィナンシャルグループは楽天銀行に出資する。5〜10%を軸に出資比率を詰めている。楽天グループは秋にも金融子会社を再編し、楽天銀の傘下に集約する。再編に合わせて他の子会社から資本を付け替える。\n楽天は10月をめどに金融事業を再編する。楽天銀を金融事業の統括会社とし、楽天カードや楽天証券ホールディングス（HD）などを傘下に入れる。これを受けてみずほが楽天銀に出資する。現在49%を出資する楽天...', '人工知能（AI）を活用した音声合成を手掛けるタイタンインテリジェンス（東京・渋谷）は、話者の感情や声質を再現して多言語に吹き替えるサービスを18日に始める。フランス語など200言語に対応し、アニメや動画コンテンツの世界展開を支援する。\nサービス名は「mimidub（ミミダブ）」で、話者の声質や話し方、感情を再現して吹き替える。タイタンインテリジェンスに吹き替え作業を依頼する形式で、1週間ほどで納...', 'ソフトバンクは広告に使うショート動画の作成を生成AI（人工知能）が代行する技術を開発した。静止画と簡単な指示で映像制作に関わる作業を代替できる。\n動画1本の制作時間は10分ほどで費用も1000円程度で済む。ネット通販やデジタルサイネージ（電子掲示板）に掲出する広告の制作などの用途を見込む。\n開発元が異なる複数のAIが自律的に連携して1本の動画をつくる。監督役のAIが静止画などから広告の意図をくみ...', '台湾積体電路製造（TSMC）の熊本工場を運営する子会社の四半期ベースの最終損益が量産開始後、初めて黒字になった。2026年1〜3月期に9億5138万台湾ドル（約47億円）の黒字だった

In [7]:
img_class = "image_i1obkbgm"
img_tags = soup.find_all(class_=img_class)
img_tags


[<img alt="ウェザーニューズの雨雲レーダーで解説するKFB福島放送の伊藤えがお気象予報士" class="image_i1obkbgm" src="https://article-image-ix.nikkei.com/https%3A%2F%2Fimgix-proxy.n8s.jp%2FDSXZQO2905509016042026000000-1.png?ixlib=js-3.8.0&amp;w=380&amp;h=237&amp;auto=format%2Ccompress&amp;fit=crop&amp;bg=FFFFFF&amp;fp-x=0.05&amp;fp-y=0.58&amp;fp-z=1&amp;crop=focalpoint&amp;s=644ad641e52a141f5530b6b9707f8de7" width="380"/>,
 <img alt="" class="image_i1obkbgm" src="https://article-image-ix.nikkei.com/https%3A%2F%2Fimgix-proxy.n8s.jp%2FDSXZQO2993360010052026000000-1.jpg?ixlib=js-3.8.0&amp;w=380&amp;h=237&amp;auto=format%2Ccompress&amp;fit=crop&amp;bg=FFFFFF&amp;fp-x=0.5&amp;fp-y=0.5&amp;fp-z=1&amp;crop=focalpoint&amp;s=a77c2b891f7bf6a47718cc1c0f3c31c2" width="163"/>,
 <img alt="AI技術で話し手の声質や感情を再現して吹き替える" class="image_i1obkbgm" src="https://article-image-ix.nikkei.com/https%3A%2F%2Fimgix-proxy.n8s.jp%2FDSXZQO3029847015052026000000-1.jpg?ixlib=js-3.8.0&amp;w=380&amp;h=237&amp;auto=format%2Ccompress&amp;fit=crop&amp;bg=FFFFFF&amp;

In [8]:

  img_urls = []

  for img_tag in img_tags:
      img_url = img_tag.get("src")
      if img_url:
          img_urls.append(img_url)

  print(img_urls)


['https://article-image-ix.nikkei.com/https%3A%2F%2Fimgix-proxy.n8s.jp%2FDSXZQO2905509016042026000000-1.png?ixlib=js-3.8.0&w=380&h=237&auto=format%2Ccompress&fit=crop&bg=FFFFFF&fp-x=0.05&fp-y=0.58&fp-z=1&crop=focalpoint&s=644ad641e52a141f5530b6b9707f8de7', 'https://article-image-ix.nikkei.com/https%3A%2F%2Fimgix-proxy.n8s.jp%2FDSXZQO2993360010052026000000-1.jpg?ixlib=js-3.8.0&w=380&h=237&auto=format%2Ccompress&fit=crop&bg=FFFFFF&fp-x=0.5&fp-y=0.5&fp-z=1&crop=focalpoint&s=a77c2b891f7bf6a47718cc1c0f3c31c2', 'https://article-image-ix.nikkei.com/https%3A%2F%2Fimgix-proxy.n8s.jp%2FDSXZQO3029847015052026000000-1.jpg?ixlib=js-3.8.0&w=380&h=237&auto=format%2Ccompress&fit=crop&bg=FFFFFF&fp-x=0.5&fp-y=0.5&fp-z=1&crop=focalpoint&s=76a1760f914bfddef2c3db86bc4dabee', 'https://article-image-ix.nikkei.com/https%3A%2F%2Fimgix-proxy.n8s.jp%2FDSXZQO2771984018032026000000-1.jpg?ixlib=js-3.8.0&w=380&h=237&auto=format%2Ccompress&fit=crop&bg=FFFFFF&fp-x=0.66&fp-y=0.3&fp-z=1&crop=focalpoint&s=2997f0e8686e6da

In [9]:
img_urls = []
img_urls = [img.get("src") for img in img_tags if img.get("src")]

print(img_urls)

['https://article-image-ix.nikkei.com/https%3A%2F%2Fimgix-proxy.n8s.jp%2FDSXZQO2905509016042026000000-1.png?ixlib=js-3.8.0&w=380&h=237&auto=format%2Ccompress&fit=crop&bg=FFFFFF&fp-x=0.05&fp-y=0.58&fp-z=1&crop=focalpoint&s=644ad641e52a141f5530b6b9707f8de7', 'https://article-image-ix.nikkei.com/https%3A%2F%2Fimgix-proxy.n8s.jp%2FDSXZQO2993360010052026000000-1.jpg?ixlib=js-3.8.0&w=380&h=237&auto=format%2Ccompress&fit=crop&bg=FFFFFF&fp-x=0.5&fp-y=0.5&fp-z=1&crop=focalpoint&s=a77c2b891f7bf6a47718cc1c0f3c31c2', 'https://article-image-ix.nikkei.com/https%3A%2F%2Fimgix-proxy.n8s.jp%2FDSXZQO3029847015052026000000-1.jpg?ixlib=js-3.8.0&w=380&h=237&auto=format%2Ccompress&fit=crop&bg=FFFFFF&fp-x=0.5&fp-y=0.5&fp-z=1&crop=focalpoint&s=76a1760f914bfddef2c3db86bc4dabee', 'https://article-image-ix.nikkei.com/https%3A%2F%2Fimgix-proxy.n8s.jp%2FDSXZQO2771984018032026000000-1.jpg?ixlib=js-3.8.0&w=380&h=237&auto=format%2Ccompress&fit=crop&bg=FFFFFF&fp-x=0.66&fp-y=0.3&fp-z=1&crop=focalpoint&s=2997f0e8686e6da

In [15]:
import pandas as pd
news_date = {
    'title': titles,
    'body_texts': body_texts,
    'img_urls': img_urls
}


# print(len(titles))
# print(len(body_texts))
# print(len(img_urls))
df = pd.DataFrame.from_dict(news_date)
df


,title,body_texts,img_urls
0,ウェザーニューズ、20万人の「天気愛」をお茶の間に,天候不順や激甚災害が増えるなか、天気情報への関心はこれまでになく高まっている。高まる関心を生...,https://article-image-ix.nikkei.com/https%3A%2...
1,みずほが楽天銀行出資へ5〜10%軸、楽天子会社再編で資本付け替え,みずほフィナンシャルグループは楽天銀行に出資する。5〜10%を軸に出資比率を詰めている。楽天...,https://article-image-ix.nikkei.com/https%3A%2...
2,「AI吹き替え」で感情や声質も再現200言語に対応、新興がサービス,人工知能（AI）を活用した音声合成を手掛けるタイタンインテリジェンス（東京・渋谷）は、話者の...,https://article-image-ix.nikkei.com/https%3A%2...
3,宣材動画の制作10分でソフトバンク、生成AIで作成代行,ソフトバンクは広告に使うショート動画の作成を生成AI（人工知能）が代行する技術を開発した。静...,https://article-image-ix.nikkei.com/https%3A%2...
4,TSMC熊本工場、初の最終黒字26年1〜3月期,台湾積体電路製造（TSMC）の熊本工場を運営する子会社の四半期ベースの最終損益が量産開始後、...,https://article-image-ix.nikkei.com/https%3A%2...
5,Claude MythosだけじゃないAIサイバー攻撃が暴く未知のバグ,米グーグルは12日、犯罪組織がAIを駆使することで開発元すら知らないソフトウエアの欠陥を発見...,https://article-image-ix.nikkei.com/https%3A%2...
6,漫画家やイラストレーター「AI普及で収入減」2割著作権保護求める声,,https://article-image-ix.nikkei.com/https%3A%2...
7,ソフトバンク系､炒め物特化の調理ロボ冷蔵庫から自動で食材,ソフトバンクグループ（SBG）傘下のソフトバンクロボティクス（東京・港）は、炒め調理に特化し...,https://article-image-ix.nikkei.com/https%3A%2...
8,GMO純利益14%増1〜3月、キャッシュレス決済やネット金融好調,GMOインターネットグループが15日発表した2026年1〜3月期の連結決算（国際会計基準）は...,https://article-image-ix.nikkei.com/https%3A%2...
9,エレコムの営業利益6%増27年3月期、純利益は反動減,エレコムは15日、2027年3月期の連結営業利益が前期比6%増の165億円になりそうだと発表...,https://article-image-ix.nikkei.com/https%3A%2...


In [16]:
df.to_csv("nikkei_news.csv", index=None, encoding="utf-8-sig")

In [23]:
import requests
from bs4 import BeautifulSoup
page_start = 1
page_end = 2
for page in range(page_start, page_end+1):
    #htmlの取得
    url = "https://www.nikkei.com/business/net-media/?page={}".format(page)
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    #データー取得-タイトル
    links= soup.find_all("a", class_="fauxBlockLink_f6t5v6i")

    link_page_one = soup.find_all("a", class_="fauxBlockLink_fzas0ps")
links.insert(0, link_page_one[0])




1
<a class="fauxBlockLink_f6t5v6i" href="/article/DGXZQOUB092PE0Z00C26A5000000/">みずほが楽天銀行出資へ　5〜10%軸、楽天子会社再編で資本付け替え</a>
2
<a class="fauxBlockLink_f6t5v6i" href="/article/DGXZQOCC12B4O0S6A510C2000000/">バンコクで「蓮活」ブーム、花束が写真映えアイテムに</a>
